# Calibration — sample, then run the posterior

This page closes the WP10 calibration path: build a `BayesianModel`, draw a
short MCMC sample, then push those draws through `posterior_runs` under a
baseline and a scenario. The claims to check:

1. NUTS recovers the infection rate that generated the synthetic data (inside
   the central 95% interval).
2. Posterior quantile ribbons for infecteds bracket the truth trajectory.
3. A scenario that raises recovery yields a negative median difference in
   infecteds at mid-run (fewer infecteds than baseline).


In [ ]:
from typing import Any, NamedTuple

import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowModel,
    OutputSet,
    Property,
    PropertyData,
    PropertyMap,
    Target,
    TargetSet,
    TransitionFlow,
    derived_refs,
)
from summer4.epi.calibration import (
    BayesianModel,
    NormalLikelihood,
    Scenario,
    Uniform,
)


## Synthetic SIR truth and targets

Compile an SIR, run it at a known infection rate, and treat sparse infecteds
samples as observations with a Normal likelihood.


In [ ]:
from summer4 import SavePlan, SaveRequest


class Rates(NamedTuple):
    infection: float
    recovery: float


state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
refs = derived_refs(Rates)
model = FlowModel(pmap)
model.add_flow(TransitionFlow("infection", state["S"], state["I"], refs.infection))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], refs.recovery))
cm = model.compile()

true = Rates(infection=0.35, recovery=0.1)
y0 = PropertyData.wrap(pmap, np.array([999.0, 1.0, 0.0]))
obs_times = np.array([0.0, 20.0, 40.0, 60.0])
qty = Compartments(where=state["I"])
outputs = OutputSet()
outputs["I"] = Compartments(where=state["I"]).total()

truth = outputs.evaluate(
    cm.run(
        true._asdict(),
        y0,
        t0=0.0,
        t1=80.0,
        dt=1.0,
        save=outputs.plan(SavePlan()),
        solver="euler",
    ),
    true._asdict(),
)
truth_vals = truth["I"].values
truth_I = np.asarray(
    truth_vals.data if hasattr(truth_vals, "data") else truth_vals
).reshape(-1)

obs = cm.run(
    true._asdict(),
    y0,
    t0=0.0,
    t1=80.0,
    dt=1.0,
    save=SavePlan(requests={"I": SaveRequest(qty, ts=obs_times)}),
    solver="euler",
)
raw = obs["I"].at_times(obs_times).values
obs_values = np.asarray(raw.data if hasattr(raw, "data") else raw).reshape(-1)

targets = TargetSet(
    targets=(
        Target(
            key="I",
            times=obs_times,
            values=obs_values,
            quantity=qty,
            likelihood=NormalLikelihood(sd=8.0),
        ),
    )
)

bm = BayesianModel(
    cm,
    {"infection": 0.2, "recovery": 0.1},
    priors=(Uniform("infection", 0.05, 0.8),),
    targets=targets,
    outputs=outputs,
    y0=y0,
    run_kwargs=dict(t0=0.0, t1=80.0, dt=1.0, solver="euler"),
)


## NUTS sample

A short chain is enough for this one-parameter SIR. The truth infection rate
must land inside the central 95% of the posterior.


In [ ]:
idata = bm.sample(
    kind="nuts",
    num_warmup=100,
    num_samples=100,
    num_chains=1,
    seed=0,
    progress_bar=False,
)
post = np.asarray(idata.posterior["infection"]).reshape(-1)
lo, mid, hi = np.quantile(post, [0.025, 0.5, 0.975])
print(f"posterior infection median={mid:.3f} 95% CI [{lo:.3f}, {hi:.3f}]")

hist = pd.DataFrame({"infection": post})
figure = hist.plot.hist(nbins=20, title="Posterior infection rate")
figure.update_layout(xaxis_title="infection", yaxis_title="count", showlegend=False)
figure.add_vline(x=true.infection, line_dash="dash", annotation_text="truth")

assert lo <= true.infection <= hi


## Posterior runs and quantile ribbons

`posterior_runs` keeps per-draw series. Quantile frames use a time index and
string-labelled quantile columns — the Kiribati uncertainty parquet schema.


In [ ]:
runs = bm.posterior_runs(
    idata,
    n=40,
    seed=1,
    scenarios={
        "baseline": None,
        "faster_recovery": Scenario(params={"recovery": 0.25}),
    },
    batch_size=16,
)
ribbons = runs.quantiles(q=(0.025, 0.5, 0.975))["baseline"]
times = np.asarray(ribbons.index)
band = pd.DataFrame(
    {
        "low": ribbons[("I", "0.025")].to_numpy(),
        "median": ribbons[("I", "0.5")].to_numpy(),
        "high": ribbons[("I", "0.975")].to_numpy(),
        "truth": truth_I,
    },
    index=times,
)
figure = band.plot(title="Infecteds — posterior median and 95% ribbon vs truth")
figure.update_layout(xaxis_title="time", yaxis_title="infecteds")

# Truth should sit inside the ribbon at mid-epidemic.
mid_t = int(np.argmin(np.abs(times - 40.0)))
assert band["low"].iloc[mid_t] <= band["truth"].iloc[mid_t] <= band["high"].iloc[mid_t]


## Scenario difference

Raising recovery shortens the infectious period. The median difference
(scenario − baseline) in infecteds at t=40 should be negative.


In [ ]:
diffs = runs.differences(
    ref="baseline",
    outputs={"I_delta": "I"},
    at=40.0,
    relative=True,
    q=(0.025, 0.5, 0.975),
)
frame = diffs["faster_recovery"]
print(frame)
figure = frame[["I_delta"]].plot.bar(title="Infecteds difference at t=40 (scenario − baseline)")
figure.update_layout(xaxis_title="quantile", yaxis_title="Δ infecteds", showlegend=False)

assert float(frame.loc["0.5", "I_delta"]) < 0.0
assert "I_delta_relative" in frame.columns
